In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

In [2]:
ratings = pd.read_csv("./ml-latest-small/ratings.csv")
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [3]:
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [4]:
user_movie = ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)
user_movie

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,0.0,0.0,0.0,0.0,0.0,2.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
607,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,2.5,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
original_matrix = user_movie.copy()

In [6]:
user_mean = user_movie.replace(0, np.nan).mean(axis=1)
centered = user_movie.sub(user_mean, axis=0).fillna(0)

In [7]:
latent_features = 50
svd = TruncatedSVD(n_components=latent_features, random_state=42)
U_sigma = svd.fit_transform(centered)
Vt = svd.components_

In [8]:
predicted = np.dot(U_sigma, Vt)

In [9]:
predicted = predicted + user_mean.values.reshape(-1, 1)

predicted_df = pd.DataFrame(
    predicted,
    index=user_movie.index,
    columns=user_movie.columns
)

In [10]:
def recommend_movies(user_id, top_n=5):

    if user_id not in original_matrix.index:
        print("User not found.")
        return

    # Movies already rated
    rated = original_matrix.loc[user_id]
    rated_movies = rated[rated > 0].index

    # Predicted ratings
    predictions = predicted_df.loc[user_id]

    # Remove rated movies
    predictions = predictions.drop(rated_movies)

    # Sort descending
    recommendations = predictions.sort_values(ascending=False)

    return recommendations.head(top_n)

In [11]:
movies = pd.read_csv("./ml-latest-small/movies.csv")

In [12]:
user_id = int(input("Enter User ID: "))
recommendations = recommend_movies(user_id)
result = recommendations.reset_index()
result.columns = ["movieId", "Predicted Rating"]
result = result.merge(movies, on="movieId")
print(result[["movieId", "title", "Predicted Rating"]])

Enter User ID:  1


   movieId                           title  Predicted Rating
0     1036                 Die Hard (1988)          4.169244
1     1221  Godfather: Part II, The (1974)          3.172933
2     1259              Stand by Me (1986)          3.007575
3      858           Godfather, The (1972)          2.975516
4     1387                     Jaws (1975)          2.934698
